# TabICLv2 Regressor — DIMER artifact inference tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabicl-regressor-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabicl-regressor-pipeline/blob/main/tutorials/tabiclv2_regressor_artifact_inference_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-jingang%2FTabICL-ffcc4d?style=flat)](https://huggingface.co/jingang/TabICL) [![Upstream](https://img.shields.io/badge/Upstream-soda--inria%2Ftabicl-181717?style=flat&logo=github&logoColor=white)](https://github.com/soda-inria/tabicl) [![arXiv](https://img.shields.io/badge/arXiv-2602.11139-b31b1b.svg)](https://arxiv.org/abs/2602.11139)

**Profile:** `ARTIFACT-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** serving-state reconstruction from an externally produced DIMER-style TabICLv2 regressor bundle (`artifact.json` + checkpoint + training context in a ZIP) and point-prediction inference on genuinely new rows

**This notebook is standalone.** It carries the repository's pipeline module (`src/tabicl_regressor_pipeline/api.py` at revision `ca6b2d9f77f9`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `4dcd344ece2c00be9e831fdd35bed57b5ad83e19` (~114 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** **Known NOTEBOOK_SPEC 2.0 gap (§19, SART1/RUN5/RUN2):** the default path does not yet obtain a trusted sample bundle or sample input automatically — with `ARTIFACT_ZIP_PATH` and `NEW_DATA_PATH` empty, Sections 4 and 6 open upload dialogs for a predictor bundle produced by the E2E tutorial and for unlabelled rows; an executor sets both paths to files already in the runtime to skip the dialogs. Until a published sample bundle and sample rows are wired in, this notebook is a `Candidate`, not release-grade. Once they are present, **Run all** installs the pinned dependencies, validates the bundle (path-safe extraction, manifest digests, provenance, pinned model identity) before any deserialisation, reconstructs the in-context regressor from the bundle alone (support rows and fitted encoders restored, nothing refit), validates the new rows into an input manifest, emits point predictions (no per-prediction uncertainty), reports what cannot be measured, and exports outputs — all inside this kernel, with no DIMER worker or service and no credential.

**Bring Your Own Data:** New-input BYOD is the `NEW_DATA_PATH`/upload branch in Section 6: your own unlabelled CSV with the bundle's required feature columns passes through the same validation, prediction and export cells. A user-supplied bundle is the separate `ARTIFACT_ZIP_PATH`/upload branch in Section 4 (`EXPECTED_ZIP_SHA256` pins it), validated before any state is reconstructed. Uploads stay inside this runtime; do not upload confidential or restricted data unless you are authorised to process it here.

This notebook consumes a serving bundle ZIP produced **outside this execution** (for example by the E2E tutorial in a separate session): it extracts it only after every member passed the path, symlink and expanded-size checks, verifies the manifest's payload allowlist, sizes and SHA-256 digests, validates the artifact/runtime provenance, rebuilds the in-context regressor from the bundle alone (bundled checkpoint + training context + recorded inference settings), accepts genuinely new unlabelled rows, predicts continuous point estimates, and exports results. **No artifact is created here** and no gradient step runs.

**Trust boundary.** Digest and manifest checks establish internal consistency, not sender authenticity, and the bundled `checkpoints/best.ckpt` is deserialised by `tabicl` (a Lightning/PyTorch checkpoint) — you are trusting the producer of the ZIP. The pinned base checkpoint of Section 3 is acquired and digest-verified independently so the manifest's `baseModelSha256` can be checked against a known-good value. Use only bundles from a trusted producer.

**Learning objectives:** install the pinned runtime, read what the carried package guarantees, resolve and digest-verify the immutable upstream checkpoint, supply an externally produced bundle and validate it before any model state is reconstructed, inspect its provenance and runtime compatibility, reconstruct the serving state from the bundle alone, validate new unlabelled rows into an input manifest, predict continuous point estimates, produce an evaluation report that is `not-measurable` because no labels exist, and export machine-readable predictions plus provenance.

**This notebook does not demonstrate:** artifact creation, in-notebook support fitting, fine-tuning, classification, or any uncertainty interval or quality claim: without labelled rows nothing is measured, and the exported predictions are point estimates with no prediction interval.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.11+). The default path runs on CPU and uses CUDA automatically when available.
- **Artifact:** an externally produced bundle ZIP (`artifact.json`, `checkpoints/best.ckpt`, `training_context.parquet`; the E2E tutorial writes `outputs/tabiclv2_regressor_artifact.zip`). Supply it through the upload dialog, or set `ARTIFACT_ZIP_PATH` to a file already present in the runtime for non-interactive execution. Nothing in this notebook manufactures it.
- **Data:** one separate, unlabelled CSV with the bundle's feature columns. It is supplied by upload or by `NEW_DATA_PATH`; no sample is bundled, because scoring self-generated rows would not be external-artifact evidence. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `jingang/TabICL` snapshot (~114 MB in total) at revision `4dcd344ece2c…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `numpy`, `pandas`, `sklearn` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'tabicl==2.1.1',
    'torch==2.11.0',
    'numpy==2.5.3',
    'pandas==2.3.3',
    'scikit-learn==1.9.0',
    'pyarrow==25.0.1',
    'lightgbm==4.7.0',
    'huggingface-hub==1.30.0',
    'transformers==5.6.2',
    'wandb==0.27.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'tabicl-regressor-pipeline',
    'repository_revision': 'ca6b2d9f77f9d60dcfeb82412e315696967c0041',
    'embedded_module': 'src/tabicl_regressor_pipeline/api.py',
    'embedded_modules': ['src/tabicl_regressor_pipeline/api.py'],
    'module_sha256': '9d19a022f6513c9aa714c000e08f8f1b3c6a7f98aaf47aa34762bfa3c6643d2f',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, numpy, pandas, sklearn
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'sklearn': sklearn.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/tabicl_regressor_pipeline/` @ `ca6b2d9f77f9`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/tabicl_regressor_pipeline/api.py`

In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib.metadata
import io
import json
import math
import os
import shutil
import stat
import sys
import zipfile
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

ARTIFACT_FORMAT = "tabicl-dimer-regressor-v1"

# Fleet snapshot identity (DIMER Notebook Specification 1.1, ST3/MOD13). The pinned upstream checkpoint is
# unchanged; these are the fleet-standard names for the same repository, revision, license and snapshot key.
# The BASE_* spellings below stay as the package's published names and alias these constants.
MODEL_ID = "jingang/TabICL"
MODEL_REVISION = "4dcd344ece2c00be9e831fdd35bed57b5ad83e19"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "tabicl-regressor-v2"
MANIFEST_NAME = "dimer-base-manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative

BASE_MODEL_REPO = MODEL_ID
BASE_CHECKPOINT_NAME = "tabicl-regressor-v2-20260212.ckpt"
BASE_MODEL_REVISION = MODEL_REVISION
BASE_MODEL_SHA256 = "0db9cb538f114e79026bf08f45f41ad8dd7ad2de2aaca9a5ca8cd3bd9748ae7a"

# Operational ceilings the tutorials enforce before any model execution (DAT22).
MIN_TRAIN_ROWS = 50  # labelled support rows required to condition the regressor
MIN_EVAL_ROWS = 2  # labelled rows required for a holdout / test partition
MAX_TRAIN_ROWS = 50_000  # support rows per conditioning call
MAX_FEATURES = 2_000  # feature columns per table
MAX_ARTIFACT_EXPANDED_BYTES = 1024 * 1024 * 1024  # 1 GiB expanded ZIP size for a serving artifact
DEFAULT_N_ESTIMATORS = 8
DEFAULT_RANDOM_STATE = 42
METRIC_IDS = ("mae", "mse", "rmse", "r2", "pearsonr")  # the ids `regression_metrics` reports


def runtime_identity() -> dict[str, str]:
    import torch

    return {
        "pythonVersion": sys.version.split()[0],
        "tabiclVersion": importlib.metadata.version("tabicl"),
        "torchVersion": torch.__version__,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }


def create_regressor(
    *,
    model_path: str | Path,
    n_estimators: int,
    random_state: int,
    device: str,
    allow_auto_download: bool = False,
):
    """Construct the supported TabICLv2 regression serving estimator."""
    from tabicl import TabICLRegressor

    return TabICLRegressor(
        model_path=str(model_path),
        allow_auto_download=allow_auto_download,
        n_estimators=n_estimators,
        random_state=random_state,
        device=device,
    )


def create_finetuned_regressor(**kwargs: Any):
    """Construct the supported TabICLv2 regression fine-tuning estimator."""
    from tabicl import FinetunedTabICLRegressor

    kwargs.setdefault("allow_auto_download", False)
    return FinetunedTabICLRegressor(**kwargs)


def fine_tune_regressor(model: Any, X: Any, y: Any, **kwargs: Any) -> Any:
    """Run the upstream fine-tuning operation through the repository API."""
    return model.fit(X, y, **kwargs)


def condition_regressor(model: Any, X: Any, y: Any) -> Any:
    """Register the serving support context required by TabICL inference."""
    model.fit(X, y)
    return model


def predict_points(model: Any, X: Any) -> np.ndarray:
    """Return finite one-dimensional regression point predictions."""
    values = np.asarray(model.predict(X), dtype=float)
    if values.ndim != 1:
        values = values.reshape(-1)
    if not np.isfinite(values).all():
        raise RuntimeError("TabICL produced non-finite point predictions")
    return values


def read_single_input(*, env_var: str, label: str) -> tuple[str, bytes]:
    """Read one user-supplied file from an explicit path or Colab upload dialog."""
    explicit = os.environ.get(env_var, "").strip()
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"{label} path does not exist: {path}")
        return path.name, path.read_bytes()

    try:
        from google.colab import files  # type: ignore
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            f"{label} requires either a Colab upload or environment variable {env_var}"
        ) from exc

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Upload exactly one {label}")
    name, payload = next(iter(uploaded.items()))
    return str(name), bytes(payload)


def download_output(path: str | Path) -> Path:
    """Download in Colab; otherwise retain the file at its explicit Jupyter path."""
    resolved = Path(path).resolve()
    try:
        from google.colab import files  # type: ignore
    except ModuleNotFoundError:
        print(f"Output retained at {resolved}")
        return resolved
    files.download(str(resolved))
    return resolved


def _base_version(value: str) -> str:
    return value.split("+")[0]


def validate_artifact_runtime(
    manifest: dict[str, Any],
    *,
    expected_artifact_format: str = ARTIFACT_FORMAT,
    expected_tabicl_version: str = "2.1.1",
    expected_torch_version: str = "2.11.0",
) -> dict[str, Any]:
    """Validate and return artifact/runtime provenance before model deserialization."""
    if manifest.get("artifactFormat") != expected_artifact_format:
        raise ValueError(f"Unsupported artifactFormat: {manifest.get('artifactFormat')}")
    if manifest.get("tabiclVersion") != expected_tabicl_version:
        raise ValueError("Artifact TabICL version does not match the supported runtime")

    observed = runtime_identity()
    if _base_version(observed["torchVersion"]) != expected_torch_version:
        raise RuntimeError(
            f"Runtime torch {observed['torchVersion']} is incompatible with the "
            f"release-verified torch {expected_torch_version}"
        )

    producer = manifest.get("runtime")
    compatibility = {
        "artifactFormat": manifest.get("artifactFormat"),
        "baseCheckpoint": manifest.get("baseCheckpoint"),
        "baseModelRevision": manifest.get("baseModelRevision"),
        "baseModelSha256": manifest.get("baseModelSha256"),
        "tabiclVersion": manifest.get("tabiclVersion"),
        "producerRuntime": producer,
        "consumerRuntime": observed,
        "policy": "TabICL exact; consumer torch base version 2.11.0; device may differ",
    }

    if producer:
        producer_torch = str(producer.get("torchVersion", ""))
        if producer_torch and _base_version(producer_torch) != expected_torch_version:
            raise RuntimeError(
                f"Artifact was produced with torch {producer_torch}; expected release family "
                f"{expected_torch_version}"
            )
    else:
        compatibility["warning"] = (
            "Legacy artifact has no runtime block; TabICL/version and digest checks apply, "
            "but producer runtime compatibility cannot be proven."
        )
    return compatibility


# ---------------------------------------------------------------------------
# Fleet snapshot scheme (NOTEBOOK_SPEC 1.1 ST3/ST4, MOD13): manifest-driven verification and staging.
# ---------------------------------------------------------------------------


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local pinned snapshot against its manifest; raise naming the first mismatch.

    The manifest is the parity anchor the standalone tutorials carry inline (ST3). The package's own
    ``BASE_MODEL_SHA256`` is not replaced by it: the manifest entry for ``BASE_CHECKPOINT_NAME`` must equal
    that constant, so the two can never diverge silently.
    """
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as handle:
        manifest = json.load(handle)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    entries = manifest.get("files", [])
    declared = {entry["path"]: entry["sha256"] for entry in entries}
    if declared.get(BASE_CHECKPOINT_NAME) != BASE_MODEL_SHA256:
        raise ValueError(
            f"manifest {BASE_CHECKPOINT_NAME} sha256 {declared.get(BASE_CHECKPOINT_NAME)!r} "
            f"!= BASE_MODEL_SHA256 {BASE_MODEL_SHA256!r}"
        )
    for entry in entries:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(repo_id=MODEL_ID, filename=relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a clone commits the manifest but
    git-ignores the checkpoint). Returns the relative paths fetched; ``verify_snapshot`` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as handle:
        manifest = json.load(handle)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


class TabICLRegressionPipeline:
    """Serving wrapper: the digest-verified pinned checkpoint behind the upstream ``TabICLRegressor``.

    ``from_pretrained`` stages and verifies the snapshot and constructs the estimator through
    ``create_regressor`` (no auto-download); ``fit`` registers the support context (in-context learning,
    no gradient training) and ``predict`` returns finite point predictions through ``predict_points``.
    """

    def __init__(
        self,
        estimator: Any,
        *,
        model_path: Path,
        n_estimators: int,
        random_state: int,
        device: str,
        source: str = "local-snapshot",
    ) -> None:
        self.estimator = estimator
        self.model_path = Path(model_path)
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.device = device
        self.source = source
        self.is_fitted = False

    @classmethod
    def from_pretrained(
        cls,
        weights_dir: str | Path | None = None,
        *,
        allow_download: bool = False,
        n_estimators: int = DEFAULT_N_ESTIMATORS,
        random_state: int = DEFAULT_RANDOM_STATE,
        device: str | None = None,
    ) -> TabICLRegressionPipeline:
        """Stage what is missing (at ``MODEL_REVISION``), re-hash every manifest entry, then build the
        estimator on the verified checkpoint. ``tabicl`` deserialises the checkpoint on the first ``fit``."""
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        if device is None:
            import torch

            device = "cuda" if torch.cuda.is_available() else "cpu"
        model_path = root / BASE_CHECKPOINT_NAME
        estimator = create_regressor(
            model_path=model_path,
            n_estimators=n_estimators,
            random_state=random_state,
            device=device,
            allow_auto_download=False,
        )
        return cls(
            estimator,
            model_path=model_path,
            n_estimators=n_estimators,
            random_state=random_state,
            device=device,
        )

    def fit(self, X: Any, y: Any) -> TabICLRegressionPipeline:
        condition_regressor(self.estimator, X, y)
        self.is_fitted = True
        return self

    def predict(self, X: Any) -> np.ndarray:
        if not self.is_fitted:
            raise RuntimeError("Pipeline is not conditioned; call fit(X, y) with the support rows first")
        return predict_points(self.estimator, X)


# ---------------------------------------------------------------------------
# Table preparation, encoding, metrics (extracted from the tutorials so both notebooks share one code path).
# ---------------------------------------------------------------------------


def _check_regression_table(
    frame: pd.DataFrame,
    target_column: str,
    *,
    min_rows: int,
    max_rows: int = MAX_TRAIN_ROWS,
    max_features: int = MAX_FEATURES,
) -> tuple[pd.DataFrame, int]:
    """The checks `prepare_regression_table` applies; returns the cleaned table and the dropped-row count."""
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("frame must be a pandas.DataFrame")
    if frame.columns.duplicated().any():
        dupes = sorted(set(frame.columns[frame.columns.duplicated()]))
        raise ValueError(f"table contains duplicate column names: {dupes}")
    if target_column not in frame.columns:
        raise KeyError(f"missing target {target_column!r}")
    out = frame.copy()
    numeric = pd.to_numeric(out[target_column], errors="coerce")
    finite = numeric.notna() & np.isfinite(numeric.to_numpy(dtype=float, na_value=np.nan))
    dropped = int((~finite).sum())
    out = out.loc[finite].copy().reset_index(drop=True)
    out[target_column] = numeric.loc[finite].to_numpy(dtype=float)
    if len(out) < min_rows:
        raise ValueError(f"need at least {min_rows} labelled rows, got {len(out)}")
    features = [column for column in out.columns if column != target_column]
    if not features:
        raise ValueError("No feature columns")
    if len(features) > max_features or len(out) > max_rows:
        raise ValueError(
            f"Operational row/feature ceiling exceeded: rows={len(out)} (MAX_TRAIN_ROWS={max_rows}), "
            f"features={len(features)} (MAX_FEATURES={max_features})"
        )
    if out[target_column].nunique() < 2:
        raise ValueError("Regression target must vary")
    return out, dropped


def prepare_regression_table(
    frame: pd.DataFrame, target_column: str, *, min_rows: int = MIN_TRAIN_ROWS
) -> tuple[pd.DataFrame, int]:
    """Coerce the target to finite floats, drop rows where that fails (the count is returned so the notebook
    can report it — DAT23), enforce the row/feature ceilings and a varying target."""
    return _check_regression_table(frame, target_column, min_rows=min_rows)


def align_to_schema(frame: pd.DataFrame, feature_columns: Sequence[str], target_column: str) -> pd.DataFrame:
    """Require the training schema exactly; order columns like train."""
    expected = set(feature_columns) | {target_column}
    if set(frame.columns) != expected:
        raise ValueError(
            f"schema does not match train; missing={sorted(expected - set(frame.columns))}, "
            f"extra={sorted(set(frame.columns) - expected)}"
        )
    return frame[[*feature_columns, target_column]].reset_index(drop=True)


def fit_categorical_encoder(frame: pd.DataFrame, feature_columns: Sequence[str]) -> dict[str, list[str]]:
    """Ordinal maps for non-numeric columns, fitted on the training split only."""
    encoders: dict[str, list[str]] = {}
    for column in feature_columns:
        series = frame[column]
        if pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series):
            continue
        encoders[column] = sorted({str(value) for value in series.dropna().unique()})
    return encoders


def apply_categorical_encoder(
    frame: pd.DataFrame, encoders: Mapping[str, Sequence[str]]
) -> tuple[pd.DataFrame, dict[str, int]]:
    """Apply training-fitted ordinal maps; unseen or missing values get the extra 'unknown' code.

    Returns the encoded frame and, per column, how many unseen values were mapped to the unknown code
    (DAT20: the notebook reports them)."""
    out = frame.copy()
    unseen_counts: dict[str, int] = {}
    for column, categories in encoders.items():
        lookup = {category: index for index, category in enumerate(categories)}
        unknown = len(categories)
        encoded, unseen = [], 0
        for value in out[column]:
            if pd.isna(value):
                encoded.append(unknown)
                continue
            key = str(value)
            if key not in lookup:
                unseen += 1
            encoded.append(lookup.get(key, unknown))
        out[column] = encoded
        if unseen:
            unseen_counts[column] = unseen
    return out, unseen_counts


def _missing_counts(frame: pd.DataFrame) -> dict[str, int]:
    return {str(column): int(n) for column, n in frame.isna().sum().items() if n > 0}


def _check_inference_frame(frame: pd.DataFrame, feature_columns: Sequence[str]) -> pd.DataFrame:
    """The checks `read_inference_csv` applies to an inference table (after the raw-header check)."""
    if frame.columns.duplicated().any():
        dupes = sorted(set(frame.columns[frame.columns.duplicated()]))
        raise ValueError(f"Inference CSV contains duplicate column names: {dupes}")
    if "prediction" in frame.columns:
        raise ValueError("Inference CSV already contains a 'prediction' column")
    missing = [column for column in feature_columns if column not in frame.columns]
    if missing:
        raise ValueError(f"Inference CSV missing features: {missing}")
    return frame


def raw_csv_header(payload: bytes) -> list[str]:
    """First non-empty CSV row, read before pandas can rename duplicate names."""
    reader = csv.reader(io.StringIO(payload.decode("utf-8-sig")))
    for row in reader:
        if row and any(cell.strip() for cell in row):
            return row
    raise ValueError("CSV has no header")


def read_csv_payload(payload: bytes, label: str) -> pd.DataFrame:
    """Read a CSV payload, refusing duplicate header names before pandas renames them (DAT16)."""
    header = raw_csv_header(payload)
    dupes = sorted({name for name in header if header.count(name) > 1})
    if dupes:
        raise ValueError(f"{label} contains duplicate column names: {dupes}")
    return pd.read_csv(io.BytesIO(payload))


def read_inference_csv(payload: bytes, feature_columns: Sequence[str]) -> pd.DataFrame:
    """Read an unlabelled inference CSV: unique header, no `prediction` column, every feature present."""
    return _check_inference_frame(read_csv_payload(payload, "Inference CSV"), feature_columns)


def regression_metrics(y_true: Any, y_pred: Any) -> dict[str, float]:
    """The repository's metric set: mae, mse, rmse (target units), r2, pearsonr (NaN if undefined)."""
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    y = np.asarray(y_true, dtype=float).reshape(-1)
    pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if y.shape != pred.shape or y.size == 0:
        raise ValueError("y_true and y_pred must be non-empty and the same length")
    mse = float(mean_squared_error(y, pred))
    out = {
        "mae": float(mean_absolute_error(y, pred)),
        "mse": mse,
        "rmse": float(np.sqrt(mse)),
        "r2": float(r2_score(y, pred)),
    }
    with np.errstate(invalid="ignore", divide="ignore"):
        corr = np.corrcoef(y, pred)[0, 1] if y.size > 1 else float("nan")
    out["pearsonr"] = float(corr) if np.isfinite(corr) else float("nan")
    return out


def training_mean_baseline(train_targets: Any, holdout_targets: Any) -> dict[str, float]:
    """The trivial baseline: always predict the training mean (mae, mse, rmse, r2; pearsonr is undefined)."""
    train = np.asarray(train_targets, dtype=float).reshape(-1)
    holdout = np.asarray(holdout_targets, dtype=float).reshape(-1)
    if train.size == 0 or holdout.size == 0 or not (np.isfinite(train).all() and np.isfinite(holdout).all()):
        raise ValueError("targets must be non-empty and finite")
    metrics = regression_metrics(holdout, np.full(holdout.shape, float(train.mean())))
    metrics.pop("pearsonr")
    return metrics


def compare_metric(candidate: Mapping[str, float], reference: Mapping[str, float], name: str) -> bool:
    """True when `candidate` beats `reference` on metric `name` (higher r2/pearsonr, lower error metrics)."""
    candidate_value, reference_value = float(candidate[name]), float(reference[name])
    if not (math.isfinite(candidate_value) and math.isfinite(reference_value)):
        raise ValueError(f"{name} unavailable for selection")
    if name in ("r2", "pearsonr"):
        return candidate_value > reference_value
    return candidate_value < reference_value


# ---------------------------------------------------------------------------
# Role stages (DAT24 / EVAL21).
# ---------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "input": "pandas.DataFrame, one row per example; numeric/categorical features plus a numeric target",
    "columns": "unique names; non-numeric feature columns are ordinal-encoded with maps fitted on training",
    "target": (
        "coerced to float; rows whose target is missing or non-finite are dropped and counted; must vary "
        "(>= 2 distinct values)"
    ),
    "train_rows": [MIN_TRAIN_ROWS, MAX_TRAIN_ROWS],
    "eval_rows": [MIN_EVAL_ROWS, None],
    "features": [1, MAX_FEATURES],
    "inference_input": "every fitted feature column present; no `prediction` column; extras pass through",
    "preprocessing": (
        "no scaling by the package; unseen or missing categorical values map to the fitted 'unknown' code; "
        "the model is conditioned on the (encoded) training rows at prediction time"
    ),
}


def validate_inputs(
    frame: pd.DataFrame,
    target_column: str | None = "target",
    *,
    feature_columns: Sequence[str] | None = None,
    min_rows: int = MIN_TRAIN_ROWS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observed table properties, verdict).

    With a ``target_column`` the table is checked exactly as ``prepare_regression_table`` checks it (the
    number of dropped non-finite-target rows is recorded, not hidden); with ``target_column=None`` it is an
    inference table checked against ``feature_columns`` exactly as ``read_inference_csv`` checks it.
    Rejection is reported by raising the same error the core function raises.
    """
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (the table's id)")
    table_id = names[0] if names else "table-0"
    if target_column is None:
        if feature_columns is None:
            raise ValueError("feature_columns is required to validate an inference table")
        checked = _check_inference_frame(frame, list(feature_columns))
        entry: dict[str, Any] = {
            "id": table_id,
            "mode": "inference",
            "rows": len(checked),
            "feature_columns": list(feature_columns),
            "extra_columns": [column for column in checked.columns if column not in feature_columns],
            "missing_value_columns": _missing_counts(checked[list(feature_columns)]),
        }
    else:
        cleaned, dropped = _check_regression_table(frame, target_column, min_rows=min_rows)
        features = [column for column in cleaned.columns if column != target_column]
        encoders = fit_categorical_encoder(cleaned, features)
        values = cleaned[target_column].to_numpy(dtype=float)
        entry = {
            "id": table_id,
            "mode": "fit",
            "rows": len(cleaned),
            "dropped_non_finite_target_rows": dropped,
            "feature_columns": features,
            "categorical_columns": sorted(encoders),
            "missing_value_columns": _missing_counts(cleaned[features]),
            "target_summary": {
                "min": float(values.min()),
                "max": float(values.max()),
                "mean": float(values.mean()),
                "distinct": int(cleaned[target_column].nunique()),
            },
        }
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [entry],
        "target_column": target_column,
        "min_rows": min_rows if target_column is not None else None,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    metrics: Mapping[str, float] | None,
    *,
    baseline: Mapping[str, float] | None = None,
    independent_test: Mapping[str, float] | None = None,
    n_holdout: int | None = None,
    n_test: int | None = None,
    target_column: str | None = None,
    selection: str | None = None,
    sample_kind: str = "sample",
    estimation: str = "single seeded random holdout; no dispersion estimate",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    ``metrics`` / ``independent_test`` are dicts from ``regression_metrics`` and ``baseline`` from
    ``training_mean_baseline``; the verdict is ``sample-sanity``. Without metrics (no labelled rows) the
    verdict is ``not-measurable`` and the report says what labelled data would make the task measurable.
    """

    def _entries(source: Mapping[str, float]) -> list[dict[str, Any]]:
        unknown = sorted(set(source) - set(METRIC_IDS))
        if unknown:
            raise ValueError(f"unknown metric ids {unknown}; regression_metrics reports {list(METRIC_IDS)}")
        units = {
            "mae": "target units",
            "mse": "target units squared",
            "rmse": "target units",
            "r2": "unitless",
            "pearsonr": "unitless",
        }
        return [
            {
                "id": metric_id,
                "value": None if not math.isfinite(float(source[metric_id])) else float(source[metric_id]),
                "units": units[metric_id],
                "higher_is_better": metric_id in ("r2", "pearsonr"),
            }
            for metric_id in METRIC_IDS
            if metric_id in source
        ]

    base: dict[str, Any] = {
        "task": "tabular regression by in-context conditioning on labelled support rows",
        "score_semantics": "continuous point predictions in target units; no per-prediction uncertainty",
        "sample_kind": sample_kind,
        "n_holdout": n_holdout,
        "n_test": n_test,
        "target_column": target_column,
        "selection": selection,
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if metrics is None:
        return {
            **base,
            "metrics": [],
            "independent_test": [],
            "verdict": "not-measurable",
            "reason": "no labelled holdout rows were supplied for the scored table",
            "needs": (
                "a labelled holdout table with a finite, varying numeric target column, scored with "
                "`regression_metrics` (mae, mse, rmse, r2, pearsonr) against `training_mean_baseline`; an "
                "independent test partition from the deployment domain for any generalisable claim"
            ),
        }
    reported = [{**entry, "estimation": estimation} for entry in _entries(metrics)]
    test_entries: list[dict[str, Any]] = []
    if independent_test is not None:
        test_estimation = "independent test partition, single run"
        test_entries = [{**e, "estimation": test_estimation} for e in _entries(independent_test)]
    baselines = [] if baseline is None else [{"id": "training_mean", "metrics": _entries(baseline)}]
    rows = "an unstated number of" if n_holdout is None else str(n_holdout)
    return {
        **base,
        "metrics": reported,
        "independent_test": test_entries,
        "baselines": baselines,
        "verdict": "sample-sanity",
        "reason": f"{rows} labelled holdout row(s) from one seeded split; tutorial evidence, not a benchmark",
        "needs": (
            "an independent, domain-representative labelled test set for any generalisable quality claim; "
            "the point predictions carry no uncertainty interval"
        ),
    }


# ---------------------------------------------------------------------------
# Serving-artifact ZIP handling shared by the producer's fresh-reload check and the companion notebook.
# ---------------------------------------------------------------------------


def safe_extract_zip(
    zip_path: str | Path, dest: str | Path, *, max_expanded_bytes: int = MAX_ARTIFACT_EXPANDED_BYTES
) -> Path:
    """Extract a ZIP member by member after every member passed the path, symlink and size checks (AINF3).

    Members are copied individually (never ``extractall``) so a member that fails a check is never written.
    """
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    root = dest.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        expanded_bytes = 0
        for info in archive.infolist():
            if "\\" in info.filename:
                raise ValueError(f"Ambiguous backslash ZIP member: {info.filename}")
            expanded_bytes += info.file_size
            if expanded_bytes > max_expanded_bytes:
                raise ValueError(f"Artifact exceeds {max_expanded_bytes} expanded bytes")
            name = info.filename
            parts = Path(name).parts
            mode = info.external_attr >> 16
            if name.startswith("/") or ".." in parts or stat.S_ISLNK(mode):
                raise ValueError(f"Unsafe ZIP member: {info.filename}")
            target = (dest / Path(name)).resolve()
            if root != target and root not in target.parents:
                raise ValueError("ZIP member escapes destination")
        for info in archive.infolist():
            target = (dest / Path(info.filename)).resolve()
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)
    return root


def manifest_member_path(root: str | Path, value: Any, field: str) -> Path:
    """Resolve a manifest path inside the bundle root, refusing absolute paths and traversal."""
    rel = Path(str(value))
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"Unsafe {field} path in artifact.json: {value!r}")
    root_resolved = Path(root).resolve()
    target = (root_resolved / rel).resolve()
    if root_resolved != target and root_resolved not in target.parents:
        raise ValueError(f"{field} path escapes artifact root: {value!r}")
    return target


def verify_artifact_bundle(root: str | Path, manifest: Mapping[str, Any]) -> dict[str, Path]:
    """Check an extracted bundle's allowlist, sizes and digests against its manifest; return member paths."""
    root = Path(root)
    ckpt = manifest_member_path(root, manifest["checkpoint"], "checkpoint")
    context_path = manifest_member_path(root, manifest["trainingContext"], "trainingContext")
    payload_files = manifest.get("payloadFiles")
    if payload_files is not None:
        expected_files = {"artifact.json", *payload_files}
        actual_files = {p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()}
        if actual_files != expected_files:
            unexpected = sorted(actual_files ^ expected_files)
            raise RuntimeError(f"Unexpected or missing artifact files: {unexpected}")
    if manifest.get("sizes"):
        if ckpt.stat().st_size != manifest["sizes"]["checkpoint"]:
            raise RuntimeError("Checkpoint size mismatch")
        if context_path.stat().st_size != manifest["sizes"]["trainingContext"]:
            raise RuntimeError("Training-context size mismatch")
    if sha256_file(ckpt) != manifest["digests"]["checkpointSha256"]:
        raise RuntimeError("Checkpoint digest mismatch")
    if sha256_file(context_path) != manifest["digests"]["trainingContextSha256"]:
        raise RuntimeError("Training-context digest mismatch")
    return {"checkpoint": ckpt, "training_context": context_path}

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `4dcd344ece2c…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TabICLRegressionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, n_estimators=8, random_state=42)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "tabicl-regressor-v2",
  "modelId": "jingang/TabICL",
  "revision": "4dcd344ece2c00be9e831fdd35bed57b5ad83e19",
  "files": [
    {
      "path": "tabicl-regressor-v2-20260212.ckpt",
      "bytes": 114324594,
      "sha256": "0db9cb538f114e79026bf08f45f41ad8dd7ad2de2aaca9a5ca8cd3bd9748ae7a"
    }
  ],
  "totalBytes": 114324594
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TabICLRegressionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, n_estimators=8, random_state=42)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Supply the external bundle and validate it before any model state is reconstructed

Leave `ARTIFACT_ZIP_PATH` empty to upload the ZIP; set it to a file already in the runtime to skip the dialog (an executor places the file there). Optionally paste the producer's ZIP SHA-256 into `EXPECTED_ZIP_SHA256` to pin the whole archive. `safe_extract_zip` extracts member by member only after every member passed the path, symlink and expanded-size checks (AINF3; it never calls `extractall`); `verify_artifact_bundle` then checks the manifest's payload allowlist, sizes and SHA-256 digests, and `validate_artifact_runtime` checks the artifact format, the TabICL version and the producer/consumer torch families (AINF4). The manifest's base-model identity is also compared with the carried package's pinned identity and digest, so a bundle built on another base model is refused. A mismatch anywhere stops the notebook.

In [ ]:
import json
import os
import shutil

ARTIFACT_ZIP_PATH = ''  # @param {type:"string"}
EXPECTED_ZIP_SHA256 = ''  # @param {type:"string"}
os.makedirs('outputs', exist_ok=True)
if ARTIFACT_ZIP_PATH:
    zip_name, zip_payload = os.path.basename(ARTIFACT_ZIP_PATH), Path(ARTIFACT_ZIP_PATH).read_bytes()
    artifact_source = f'path: {ARTIFACT_ZIP_PATH}'
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one bundle ZIP')
    zip_name, zip_payload = next(iter(uploaded.items()))
    artifact_source = 'upload dialog'
zip_path = Path('external-artifact') / Path(zip_name).name
zip_path.parent.mkdir(parents=True, exist_ok=True)
zip_path.write_bytes(zip_payload)
zip_sha256 = sha256_file(zip_path)
if EXPECTED_ZIP_SHA256:
    expected = EXPECTED_ZIP_SHA256.strip().lower()
    if len(expected) != 64 or any(character not in '0123456789abcdef' for character in expected):
        raise ValueError('EXPECTED_ZIP_SHA256 must be 64 hex chars')
    if zip_sha256 != expected:
        raise RuntimeError('Artifact ZIP SHA-256 mismatch')
extract_dir = Path('external-artifact') / 'bundle'
if extract_dir.exists():
    shutil.rmtree(extract_dir)
safe_extract_zip(zip_path, extract_dir)
matches = list(extract_dir.rglob('artifact.json'))
if len(matches) != 1:
    raise ValueError('Expected exactly one artifact.json')
bundle_root = matches[0].parent
manifest = json.loads(matches[0].read_text(encoding='utf-8'))
compatibility = validate_artifact_runtime(manifest, expected_tabicl_version=importlib.metadata.version('tabicl'), expected_torch_version=torch.__version__.split('+')[0])
if (manifest.get('baseCheckpoint'), manifest.get('baseModelRevision'), manifest.get('baseModelSha256')) != (BASE_CHECKPOINT_NAME, MODEL_REVISION, BASE_MODEL_SHA256):
    raise RuntimeError('Bundle was not produced on the pinned base checkpoint carried by this notebook')
members = verify_artifact_bundle(bundle_root, manifest)
FEATURE_COLUMNS = list(manifest['featureColumns'])
TARGET_COLUMN = manifest['targetColumn']
inference = manifest['inference']
print({'artifact_source': artifact_source, 'zip': zip_name, 'zip_sha256': zip_sha256, 'mode': manifest.get('mode'), 'selection': manifest.get('selectionBasis'), 'base': manifest.get('baseCheckpoint'), 'base_revision': str(manifest.get('baseModelRevision'))[:12]})
print(json.dumps(compatibility, indent=2, sort_keys=True))
print({'featureColumns': FEATURE_COLUMNS, 'targetColumn': TARGET_COLUMN, 'nEstimators': inference['nEstimators'], 'randomState': inference['randomState'], 'categoricalEncoders': sorted(inference.get('categoricalEncoders', {}))})

## 5. Reconstruct the in-context regressor from the bundle alone

The bundle's `inference` block records how the producer ran the model: ensemble size, random seed and the categorical encoders fitted on the training split. The regressor is built through `create_regressor` on the **bundled** checkpoint (no auto-download, no network fallback — AINF5/AINF6) and conditioned on the bundled training context (`fit` registers the rows; nothing is trained). The pinned base checkpoint verified in Section 3 (`pipe`) is not used for inference; it exists so the manifest's base-model digest could be checked against a known-good value. This is serving-state reconstruction, not training.

In [ ]:
context = pd.read_parquet(members['training_context'])
serving = TabICLRegressionPipeline(create_regressor(model_path=members['checkpoint'], allow_auto_download=False, n_estimators=inference['nEstimators'], random_state=inference['randomState'], device=pipe.device), model_path=members['checkpoint'], n_estimators=inference['nEstimators'], random_state=inference['randomState'], device=pipe.device, source='artifact')
serving.fit(context[FEATURE_COLUMNS], context[TARGET_COLUMN])
print({'context_rows': len(context), 'features': len(FEATURE_COLUMNS), 'device': serving.device, 'source': serving.source, 'checkpoint': str(members['checkpoint'])})

## 6. Supply new unlabelled rows → validate → input manifest

Leave `NEW_DATA_PATH` empty to upload one CSV, or set it to a file already in the runtime. The CSV must contain the bundle's feature columns (order does not matter; extra columns are preserved in the output and not passed to the model); duplicate header names, missing features, or a pre-existing `prediction` column stop the run. `validate_inputs(..., target_column=None, feature_columns=...)` is the package's public validation stage for inference tables: it applies exactly the checks `read_inference_csv` applies and returns an **input manifest** naming the schema, the row count, the extra columns and the missing-value columns; it is written to `outputs/tabiclv2_regressor_artifact_inference_input_manifest.json`. To show what rejection looks like, the cell also validates a probe with one feature column removed and records the package's own error message as a finding. The producer's categorical encoders are then applied; unseen values map to the fitted 'unknown' code and are counted.

In [ ]:
NEW_DATA_PATH = ''  # @param {type:"string"}
if NEW_DATA_PATH:
    input_name, payload = os.path.basename(NEW_DATA_PATH), Path(NEW_DATA_PATH).read_bytes()
else:
    new_upload = files.upload()
    if len(new_upload) != 1:
        raise ValueError('Upload exactly one CSV')
    input_name, payload = next(iter(new_upload.items()))
print({'ceilings': {'MIN_TRAIN_ROWS': MIN_TRAIN_ROWS, 'MAX_TRAIN_ROWS': MAX_TRAIN_ROWS, 'MAX_FEATURES': MAX_FEATURES}, 'required_features': FEATURE_COLUMNS})
rows = read_inference_csv(payload, FEATURE_COLUMNS)
input_manifest = validate_inputs(rows, None, feature_columns=FEATURE_COLUMNS, names=[input_name])
# Demonstrate rejection on a probe that breaks the fitted schema; the finding is recorded, not swallowed.
try:
    validate_inputs(rows.drop(columns=[FEATURE_COLUMNS[0]]), None, feature_columns=FEATURE_COLUMNS)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'missing-column-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/tabiclv2_regressor_artifact_inference_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))
X_new, unseen_new = apply_categorical_encoder(rows[FEATURE_COLUMNS], inference.get('categoricalEncoders', {}))
if unseen_new:
    print('unseen categorical values mapped to the unknown code:', unseen_new)

## 7. Predict, report what cannot be measured, and export

`predict` returns **continuous point estimates only** in the target's units — no prediction interval is produced, so any tolerance band is the caller's to set on labelled data. `evaluation_report` is the package's public evaluation stage and is produced even here: with no labelled rows its verdict is `not-measurable` and it states what labelled data would make the task measurable; it is written to `outputs/tabiclv2_regressor_artifact_inference_evaluation_report.json`. The prediction CSV keeps every input column plus `prediction`, and the result JSON records the externally supplied bundle identity (ZIP digest, manifest), the compatibility block, the input manifest, the notebook's source, the pinned model identity, revision and licence, and the runtime identity; it contains no credentials.

In [ ]:
out = rows.copy()
out['prediction'] = serving.predict(X_new)
out.to_csv('outputs/tabiclv2_regressor_artifact_inference_predictions.csv', index=False)
report = evaluation_report(None, n_holdout=0, target_column=TARGET_COLUMN, sample_kind='BYOD')
with open('outputs/tabiclv2_regressor_artifact_inference_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
payload = {
    'predictions': out.to_dict(orient='records'),
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'artifact': {'source': artifact_source, 'zip': zip_name, 'zip_sha256': zip_sha256, 'manifest': manifest, 'compatibility': compatibility, 'context_rows': len(context)},
    'inference': {'nEstimators': inference['nEstimators'], 'randomState': inference['randomState'], 'output': 'continuous point predictions in target units', 'uncertaintyInterval': None, 'unseenCategoricalValues': unseen_new},
    'input': {'filename': input_name, 'rows': len(out), 'features': FEATURE_COLUMNS},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'tabicl': importlib.metadata.version('tabicl'), 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'sklearn': sklearn.__version__, 'device': serving.device},
}
with open('outputs/tabiclv2_regressor_artifact_inference_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(out.head())
print(json.dumps(report, indent=2))
print(sorted(os.listdir('outputs')))

## Interpretation and limits

A successful run proves that the supplied archive passed the package's path, symlink and expanded-size checks, that its declared payload matches the manifest's allowlist, sizes and digests, that the bundle names the pinned base checkpoint carried by this notebook, that the runtime is compatible with the producer's, that an in-context regressor was rebuilt from the bundle alone, and that schema-compatible new rows were scored as continuous point estimates — without the repository being reachable. It does **not** authenticate the producer or establish predictive quality, robustness, calibration, fairness, or production fitness; the evaluation report says `not-measurable` because no labels exist here, and the exported point estimates carry no uncertainty interval. Never bypass a failed archive, manifest, digest, base-model or schema check; obtain a correct trusted bundle.

Successful execution proves that the recorded repository revision's package, carried in this notebook, can acquire and digest-verify the pinned checkpoint, validate and reconstruct an external bundle, validate the supplied inference table, execute the public prediction path and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** score rows with a deliberately unseen categorical value and inspect the unknown-code count; compare predictions across bundles exported with `mode` pretrained versus fine-tuned; hand a labelled copy of the same rows to `regression_metrics` in the E2E tutorial to obtain a `sample-sanity` report.

## References

- Repository README: https://github.com/kurtvalcorza/tabicl-regressor-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/tabicl-regressor-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/tabicl-regressor-pipeline/blob/main/docs/WEIGHTS.md
- E2E companion (produces the bundle): https://github.com/kurtvalcorza/tabicl-regressor-pipeline/blob/main/tutorials/tabiclv2_regressor_colab.ipynb
- Upstream model: https://huggingface.co/jingang/TabICL
- Upstream library: https://github.com/soda-inria/tabicl
- TabICLv2 paper: https://arxiv.org/abs/2602.11139